In [ ]:
!pip install nltk

In [ ]:
!pip install rouge_score

In [ ]:
!pip install evaluate

In [ ]:
!pip install bert_score

In [ ]:
import os
import json
import torch
import numpy as np
from tqdm import tqdm
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoProcessor
from transformers import Qwen2_5_VLProcessor, Qwen2_5_VLForConditionalGeneration
from PIL import Image
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import evaluate
import pandas as pd
import sys

# Add the process_vision_info function from training script
def process_vision_info(conversation):
    image_inputs = []
    video_inputs = []
    
    for message in conversation:
        if message["role"] == "user":
            for item in message["content"]:
                if item["type"] == "image":
                    image_inputs.append(item["image"])
                elif item["type"] == "video":
                    video_inputs.append(item["video"])
    
    return image_inputs, video_inputs

def load_model_and_processor(model_dir):
    """Load the model and processor from the given directory."""
    print(f"Loading model and processor from {model_dir}...")
    
    # Set the device (GPU if available, else CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Get list of files in model_dir to debug
    try:
        model_files = os.listdir(model_dir)
        print(f"Files in model directory: {model_files}")
    except Exception as e:
        print(f"Error listing model directory: {e}")
    
    # Load the processor
    processor = None
    try:
        print("Attempting to load processor from base model Qwen/Qwen2.5-VL-3B-Instruct...")
        processor = Qwen2_5_VLProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
        print("Successfully loaded processor from 'Qwen/Qwen2.5-VL-3B-Instruct'")
    except Exception as e:
        print(f"Error loading processor from base model: {e}")
        try:
            print("Attempting to load processor with AutoProcessor...")
            processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", 
                                                     trust_remote_code=True)
            print("Successfully loaded processor with AutoProcessor")
        except Exception as e2:
            print(f"Error loading processor with AutoProcessor: {e2}")
            print("Critical error: Could not load processor from any source")
            sys.exit(1)
    
    # Load the model
    model = None
    try:
        print("Attempting to load model with Qwen2_5_VLForConditionalGeneration...")
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_dir,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None,
            trust_remote_code=True
        )
        print("Successfully loaded model with Qwen2_5_VLForConditionalGeneration")
    except Exception as e:
        print(f"Error loading with Qwen2_5_VLForConditionalGeneration: {e}")
        try:
            print("Attempting to load model with AutoModelForCausalLM and trust_remote_code...")
            model = AutoModelForCausalLM.from_pretrained(
                model_dir,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map="auto" if device == "cuda" else None,
                trust_remote_code=True
            )
            print("Successfully loaded model with AutoModelForCausalLM")
        except Exception as e2:
            print(f"Error loading with AutoModelForCausalLM: {e2}")
            print("Critical error: Could not load model")
            sys.exit(1)
    
    if processor is None or model is None:
        print("Critical error: Failed to load model or processor")
        sys.exit(1)
        
    return model, processor, device

def load_test_data(annotation_path):
    """Load the test data from the given annotation file."""
    print(f"Loading test data from {annotation_path}...")
    
    with open(annotation_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    samples = []
    for item in test_data:
        # Get the image filename
        image_filename = item['image']
        
        # Extract question-answer pairs from conversations
        conversations = item.get('conversations', [])
        
        # Process each human-gpt pair in conversations
        for i in range(0, len(conversations), 2):
            # Make sure we have a complete pair
            if i + 1 < len(conversations):
                human_msg = conversations[i]
                gpt_msg = conversations[i + 1]
                
                # Verify the message roles
                if human_msg.get('from') == 'human' and gpt_msg.get('from') == 'gpt':
                    question = human_msg.get('value', '')
                    answer = gpt_msg.get('value', '')
                    
                    samples.append({
                        'image_filename': image_filename,
                        'question': question,
                        'reference': answer,
                        'conversation': conversations
                    })
    
    print(f"Loaded {len(samples)} test samples")
    return samples

def run_inference(model, processor, samples, images_dir, device, batch_size=1):
    """Run inference on the test samples."""
    print("Running inference...")
    
    predictions = []
    references = []
    
    # Process samples (for simplicity, we'll do this one by one)
    for sample in tqdm(samples):
        image_path = os.path.join(images_dir, sample['image_filename'])
        question = sample['question']
        reference = sample['reference']
        
        try:
            # Load the image
            image = Image.open(image_path).convert("RGB")
            
            # Prepare conversation for vision processing
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "image": image,
                        },
                        {
                            "type": "text",
                            "text": question
                        }
                    ]
                }
            ]
            
            # Process the image - we only need the image inputs for our case
            image_inputs, _ = process_vision_info(messages)
            
            # Format the conversation with the chat template
            prompt_text = processor.tokenizer.apply_chat_template(
                messages, 
                tokenize=False,
                add_generation_prompt=True
            )
            
            # Process all inputs - only pass images, not videos
            # This avoids the index error in the video processing
            model_inputs = processor(
                text=[prompt_text],
                images=image_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True
            )
            
            # Move to device
            model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
            
            # Generate the answer
            with torch.no_grad():
                outputs = model.generate(
                    **model_inputs,
                    max_new_tokens=100,
                    min_new_tokens=1,
                    do_sample=False
                )
            
            # Decode only the generated text (exclude the prompt)
            input_length = model_inputs["input_ids"].shape[1]
            response = processor.tokenizer.decode(
                outputs[0][input_length:], 
                skip_special_tokens=True
            ).strip()
            
            # Save the prediction and reference
            predictions.append(response)
            references.append(reference)
            
        except Exception as e:
            print(f"Error processing sample {sample['image_filename']}: {e}")
            # Log more details about the error
            import traceback
            traceback.print_exc()
            # In case of error, add an empty prediction to maintain alignment
            predictions.append("")
            references.append(reference)
    
    return predictions, references

def compute_metrics(predictions, references):
    """Compute evaluation metrics."""
    print("Computing metrics...")
    
    # 1. Accuracy (exact match)
    exact_matches = [1 if pred.strip() == ref.strip() else 0 for pred, ref in zip(predictions, references)]
    accuracy = sum(exact_matches) / len(exact_matches)
    
    # 2. BLEU-4
    smoothing = SmoothingFunction().method1
    bleu_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = pred.lower().split()
        ref_tokens = [ref.lower().split()]
        if len(pred_tokens) == 0:
            pred_tokens = [""]  # Avoid empty prediction error
        bleu_score = sentence_bleu(ref_tokens, pred_tokens, 
                                  weights=(0.25, 0.25, 0.25, 0.25), 
                                  smoothing_function=smoothing)
        bleu_scores.append(bleu_score)
    bleu4 = np.mean(bleu_scores)
    
    # 3. ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)['rougeL'].fmeasure for pred, ref in zip(predictions, references)]
    rouge_l = np.mean(rouge_scores)
    
    # 4. BERTScore
    bertscore = evaluate.load('bertscore')
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")
    bertscore_f1 = np.mean(bert_results['f1'])
    
    metrics = {
        'accuracy': accuracy,
        'bleu4': bleu4,
        'rouge_l': rouge_l,
        'bertscore_f1': bertscore_f1
    }
    
    return metrics

def save_results(predictions, references, samples, output_file):
    """Save the predictions and references to a file."""
    print(f"Saving results to {output_file}...")
    
    results = []
    for i, (pred, ref, sample) in enumerate(zip(predictions, references, samples)):
        results.append({
            'id': i,
            'image': sample['image_filename'],
            'question': sample['question'],
            'prediction': pred,
            'reference': ref,
            'correct': 1 if pred.strip() == ref.strip() else 0
        })
    
    df = pd.DataFrame(results)
    
    # Save to CSV
    df.to_csv(output_file, index=False)
    
    print(f"Results saved to {output_file}")

def main():
    # Define base directory (adjust this as needed)
    # For JupyterLab, you might need to use an absolute path or path relative to where the notebook is running
    base_dir = os.path.expanduser("~")  # Home directory
    
    # Paths 
    # If you're running this in JupyterLab, you might need to adjust these paths
    model_dir = "./qwen2.5-vl-finetuned3-spine"  # This seems to be working
    
    # For the test data, use os.path.join to create proper paths
    work_dir = os.path.join(base_dir, "work") if os.path.exists(os.path.join(base_dir, "work")) else "/work"
    
    # Try different potential paths for the test data
    potential_image_dirs = [
        os.path.join(work_dir, "RadSpineXR/Test/Unannoated_images_150"),
        os.path.join(work_dir, "RadSpineXR/Test/Unannotated_images_150"),  # Check for typo
        os.path.join(work_dir, "RadSpineXR", "Test", "Unannoated_images_150"),
        "../RadSpineXR/Test/Unannoated_images_150",
        "/work/RadSpineXR/Test/Unannoated_images_150"
    ]
    
    potential_annotation_paths = [
        os.path.join(work_dir, "RadSpineXR/Test/sample_150.json"),
        os.path.join(work_dir, "RadSpineXR", "Test", "sample_150.json"),
        "../RadSpineXR/Test/sample_150.json",
        "/work/RadSpineXR/Test/sample_150.json"
    ]
    
    # Find the first valid image directory
    images_dir = None
    for path in potential_image_dirs:
        if os.path.exists(path) and os.path.isdir(path):
            images_dir = path
            print(f"Found images directory at: {images_dir}")
            break
    
    if images_dir is None:
        print("Error: Could not find the images directory. Please provide the correct path:")
        print("Available directories in the current location:")
        print(os.listdir("."))
        if os.path.exists(work_dir):
            print(f"Available directories in {work_dir}:")
            print(os.listdir(work_dir))
        sys.exit(1)
    
    # Find the first valid annotation file
    annotation_path = None
    for path in potential_annotation_paths:
        if os.path.exists(path) and os.path.isfile(path):
            annotation_path = path
            print(f"Found annotation file at: {annotation_path}")
            break
    
    if annotation_path is None:
        print("Error: Could not find the annotation file. Please provide the correct path.")
        sys.exit(1)
    
    # Output file path
    output_file = "./inference_results2.csv"
    
    # Load model and processor
    model, processor, device = load_model_and_processor(model_dir)
    
    # Load test data
    samples = load_test_data(annotation_path)
    
    # Run inference
    predictions, references = run_inference(model, processor, samples, images_dir, device)
    
    # Compute metrics
    metrics = compute_metrics(predictions, references)
    
    # Save results
    save_results(predictions, references, samples, output_file)
    
    # Print metrics
    print("\n=== Evaluation Metrics ===")
    print(f"Accuracy: {metrics['accuracy']:.2f}")
    print(f"BLEU-4: {metrics['bleu4']:.2f}")
    print(f"ROUGE-L: {metrics['rouge_l']:.2f}")
    print(f"BERTScore (F1): {metrics['bertscore_f1']:.2f}")

if __name__ == "__main__":
    main()
